<a href="https://www.kaggle.com/code/ab0y04/skin-lesion-imagenet?scriptVersionId=342791841" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ===== FULL REBUILD (new session) + STAGE 16 FOLD 3: CUSTOM CNN + EFFICIENTNETB0 =====
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'   # MUST precede tensorflow import
import random, gc
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42  # fixed globally across ALL folds and ALL architectures
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")
assert 'tf_keras' in tf.keras.__name__, "STOP: Keras 3 active, not legacy. Restart before continuing."

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, MaxPooling2D, Dropout, GlobalAveragePooling2D, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, recall_score, confusion_matrix
print("Imports ready")

FINAL_CLASSES = ['bcc', 'bkl', 'df', 'melanoma', 'nevus', 'vasc']
NUM_CLASSES = 6
IMG_SIZE, BATCH_SIZE = 224, 32
N_FOLDS, CURRENT_FOLD = 5, 3
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)
print(f"Config loaded, CURRENT_FOLD = {CURRENT_FOLD}")

CV_ASSIGN_PATH = '/kaggle/input/datasets/ab0y04/cvfoldassignments/cv_fold_assignments.csv'
assert os.path.exists(CV_ASSIGN_PATH), f"STOP: file not found at {CV_ASSIGN_PATH}, check the dataset is attached"
cv_assignments = pd.read_csv(CV_ASSIGN_PATH)
print(f"Loaded cv_fold_assignments.csv: {len(cv_assignments):,} rows (expect 23,836)")
assert len(cv_assignments) == 23836, "STOP: row count mismatch"

def build_fold_split(cv_assignments, fold_num, seed=42):
    test_df = cv_assignments[cv_assignments['fold'] == fold_num].reset_index(drop=True)
    remaining = cv_assignments[cv_assignments['fold'] != fold_num].reset_index(drop=True)
    remaining = remaining.copy()
    fallback = pd.Series('unlinked_' + remaining.index.astype(str), index=remaining.index)
    remaining['_split_key'] = remaining['group_id'].fillna(fallback)
    groups = remaining.groupby('_split_key')['label'].first().reset_index()
    tr_groups, va_groups = train_test_split(groups, test_size=0.15, stratify=groups['label'], random_state=seed)
    train_df = remaining[remaining['_split_key'].isin(tr_groups['_split_key'])].drop(columns=['_split_key']).reset_index(drop=True)
    val_df = remaining[remaining['_split_key'].isin(va_groups['_split_key'])].drop(columns=['_split_key']).reset_index(drop=True)
    return train_df, val_df, test_df

train_df, val_df, test_df = build_fold_split(cv_assignments, CURRENT_FOLD, seed=SEED)
print(f"\n===== FOLD {CURRENT_FOLD} SPLIT =====")
print(f"Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}")
print(f"Proportions: train {len(train_df)/len(cv_assignments)*100:.1f}% | val {len(val_df)/len(cv_assignments)*100:.1f}% | test {len(test_df)/len(cv_assignments)*100:.1f}%")

test_groups = set(test_df['group_id'].dropna())
train_groups = set(train_df['group_id'].dropna())
val_groups = set(val_df['group_id'].dropna())
assert test_groups.isdisjoint(train_groups) and test_groups.isdisjoint(val_groups) and train_groups.isdisjoint(val_groups), \
    f"STOP: FOLD {CURRENT_FOLD} LEAKAGE detected"
print(f"Fold {CURRENT_FOLD} leakage check: PASS")

cls = np.array(FINAL_CLASSES)
cw = compute_class_weight('balanced', classes=cls, y=train_df['label'])
w_map = {c: w for c, w in zip(FINAL_CLASSES, cw)}
train_df['sample_weight'] = train_df['label'].map(w_map)
print(f"Fold {CURRENT_FOLD} class_weight:", {c: round(w,3) for c,w in zip(cls, cw)})

def make_fold_gens(preprocess_fn):
    if preprocess_fn is None:
        train_idg = ImageDataGenerator(rescale=1./255, **AUG)
        eval_idg  = ImageDataGenerator(rescale=1./255)
    else:
        train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
        eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='label', target_size=(IMG_SIZE,IMG_SIZE),
                  batch_size=BATCH_SIZE, class_mode='categorical', classes=FINAL_CLASSES)
    tr = train_idg.flow_from_dataframe(train_df, shuffle=True, seed=SEED, weight_col='sample_weight', **common)
    va = eval_idg.flow_from_dataframe(val_df, shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(test_df, shuffle=False, **common)
    return tr, va, te

def build_custom_cnn(num_classes=6, shape=(224,224,3)):
    return Sequential([
        Input(shape=shape),
        Conv2D(32,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(32,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        Conv2D(64,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(64,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        Conv2D(128,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(128,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        GlobalAveragePooling2D(),
        Dense(256,activation='relu'), Dropout(0.5),
        Dense(num_classes,activation='softmax')
    ])

def build_pretrained(base_class, num_classes=6, shape=(224,224,3)):
    base = base_class(include_top=False, weights='imagenet', input_shape=shape)
    model = Sequential([base, GlobalAveragePooling2D(), Dense(256,activation='relu'),
                          Dropout(0.3), Dense(num_classes,activation='softmax')])
    return model, base

def macro_specificity(y_true, y_pred, n_classes):
    cm = confusion_matrix(y_true, y_pred, labels=range(n_classes))
    total = cm.sum(); specs = []
    for i in range(n_classes):
        tp = cm[i,i]; fn = cm[i,:].sum()-tp; fp = cm[:,i].sum()-tp
        tn = total-tp-fn-fp
        specs.append(tn/(tn+fp) if (tn+fp)>0 else np.nan)
    return np.nanmean(specs)

print(f"\n===== Fold {CURRENT_FOLD} setup verified. Training Custom CNN + EfficientNetB0. =====\n")

FOLD1_ACC = {'custom': 0.47734326505276226, 'eff': 0.7200496585971446}
FOLD2_ACC = {'custom': 0.563886606409203, 'eff': 0.7333607230895645}
fold3_results = []

# ---------- CUSTOM CNN ----------
try:
    print(f"{'='*60}\nFOLD {CURRENT_FOLD}: Custom CNN\n{'='*60}")
    tr, va, te = make_fold_gens(None)
    print("class_indices:", te.class_indices)

    model = build_custom_cnn()
    model.compile(Adam(1e-3), 'categorical_crossentropy', ['accuracy'])
    cbs = [EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
           ModelCheckpoint(f'/kaggle/working/cv_f{CURRENT_FOLD}_custom.keras', monitor='val_accuracy', save_best_only=True),
           CSVLogger(f'/kaggle/working/cv_f{CURRENT_FOLD}_custom_log.csv', append=False)]
    model.fit(tr, validation_data=va, epochs=60, callbacks=cbs, verbose=1)

    y_true = np.asarray(te.classes)
    y_prob = model.predict(te, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    np.savez(f'/kaggle/working/cv_f{CURRENT_FOLD}_preds_custom.npz', y_true=y_true, y_pred=y_pred, y_prob=y_prob)

    reloaded = load_model(f'/kaggle/working/cv_f{CURRENT_FOLD}_custom.keras')
    verify_acc = reloaded.evaluate(te, verbose=0)[1]
    live_acc = accuracy_score(y_true, y_pred)
    print(f"Checkpoint verify: reloaded acc {verify_acc:.4f} vs live acc {live_acc:.4f}  match={abs(verify_acc-live_acc)<1e-3}")
    del reloaded

    result_row = dict(fold=CURRENT_FOLD, arch='custom', accuracy=live_acc,
        macro_f1=f1_score(y_true,y_pred,average='macro'),
        macro_auc=roc_auc_score(np.eye(NUM_CLASSES)[y_true], y_prob, average='macro', multi_class='ovr'),
        macro_sensitivity=recall_score(y_true,y_pred,average='macro'),
        macro_specificity=macro_specificity(y_true,y_pred,NUM_CLASSES), n_test=len(y_true))
    fold3_results.append(result_row)

    prior_mean = np.mean([FOLD1_ACC['custom'], FOLD2_ACC['custom']])
    deviation = abs(live_acc - prior_mean) * 100
    flag = "  <-- FLAG: deviates >5pp from folds 1-2 mean" if deviation > 5 else "  (within normal range)"
    print(f"\nFold {CURRENT_FOLD} vs Folds 1-2 mean: {live_acc:.4f} vs {prior_mean:.4f}, deviation {deviation:.1f}pp{flag}")
    print(f"RESULT: {result_row}")
    pd.DataFrame([result_row]).to_csv(f'/kaggle/working/cv_f{CURRENT_FOLD}_custom_result.csv', index=False)
    del model; gc.collect(); tf.keras.backend.clear_session()
except Exception as e:
    print(f"!!! FOLD {CURRENT_FOLD} Custom CNN FAILED: {type(e).__name__}: {e}")
    import traceback; traceback.print_exc()
    gc.collect(); tf.keras.backend.clear_session()

# ---------- EFFICIENTNETB0 ----------
try:
    print(f"\n{'='*60}\nFOLD {CURRENT_FOLD}: EfficientNetB0\n{'='*60}")
    tr, va, te = make_fold_gens(eff_pre)
    print("class_indices:", te.class_indices)

    model, base = build_pretrained(EfficientNetB0)
    log_file = f'/kaggle/working/cv_f{CURRENT_FOLD}_eff_log.csv'

    base.trainable = False
    model.compile(Adam(1e-3), 'categorical_crossentropy', ['accuracy'])
    model.fit(tr, validation_data=va, epochs=10, callbacks=[CSVLogger(log_file, append=False)], verbose=1)
    print("Phase 1 sanity:", model.evaluate(te, verbose=0))

    base.trainable = True
    model.compile(Adam(1e-5), 'categorical_crossentropy', ['accuracy'])
    cbs = [EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
           ModelCheckpoint(f'/kaggle/working/cv_f{CURRENT_FOLD}_eff.keras', monitor='val_accuracy', save_best_only=True),
           CSVLogger(log_file, append=True)]
    model.fit(tr, validation_data=va, epochs=60, callbacks=cbs, verbose=1)

    y_true = np.asarray(te.classes)
    y_prob = model.predict(te, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    np.savez(f'/kaggle/working/cv_f{CURRENT_FOLD}_preds_eff.npz', y_true=y_true, y_pred=y_pred, y_prob=y_prob)

    reloaded = load_model(f'/kaggle/working/cv_f{CURRENT_FOLD}_eff.keras')
    verify_acc = reloaded.evaluate(te, verbose=0)[1]
    live_acc = accuracy_score(y_true, y_pred)
    print(f"Checkpoint verify: reloaded acc {verify_acc:.4f} vs live acc {live_acc:.4f}  match={abs(verify_acc-live_acc)<1e-3}")
    del reloaded

    result_row = dict(fold=CURRENT_FOLD, arch='eff', accuracy=live_acc,
        macro_f1=f1_score(y_true,y_pred,average='macro'),
        macro_auc=roc_auc_score(np.eye(NUM_CLASSES)[y_true], y_prob, average='macro', multi_class='ovr'),
        macro_sensitivity=recall_score(y_true,y_pred,average='macro'),
        macro_specificity=macro_specificity(y_true,y_pred,NUM_CLASSES), n_test=len(y_true))
    fold3_results.append(result_row)

    prior_mean = np.mean([FOLD1_ACC['eff'], FOLD2_ACC['eff']])
    deviation = abs(live_acc - prior_mean) * 100
    flag = "  <-- FLAG: deviates >5pp from folds 1-2 mean" if deviation > 5 else "  (within normal range)"
    print(f"\nFold {CURRENT_FOLD} vs Folds 1-2 mean: {live_acc:.4f} vs {prior_mean:.4f}, deviation {deviation:.1f}pp{flag}")
    print(f"RESULT: {result_row}")
    pd.DataFrame([result_row]).to_csv(f'/kaggle/working/cv_f{CURRENT_FOLD}_eff_result.csv', index=False)
    del model, base; gc.collect(); tf.keras.backend.clear_session()
except Exception as e:
    print(f"!!! FOLD {CURRENT_FOLD} EfficientNetB0 FAILED: {type(e).__name__}: {e}")
    import traceback; traceback.print_exc()
    gc.collect(); tf.keras.backend.clear_session()

print(f"\n{'='*60}\nFOLD {CURRENT_FOLD} PARTIAL SUMMARY (custom + eff)\n{'='*60}")
for r in fold3_results:
    print(f"{r['arch']:8} acc={r['accuracy']:.4f}  macro_f1={r['macro_f1']:.4f}  macro_auc={r['macro_auc']:.4f}")
print(f"\nCompleted this run: {len(fold3_results)}/2. Still needed for fold {CURRENT_FOLD}: mob, res.")
print(">>> DOWNLOAD BOTH cv_f3_custom_result.csv AND cv_f3_eff_result.csv NOW. <<<")

2026-08-16 14:52:34.540237: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1786891954.736892      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786891954.799445      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1786891955.290438      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786891955.290479      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786891955.290481      24 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras
Imports ready
Config loaded, CURRENT_FOLD = 3
Loaded cv_fold_assignments.csv: 23,836 rows (expect 23,836)

===== FOLD 3 SPLIT =====
Train 16,230 | Val 2,852 | Test 4,754
Proportions: train 68.1% | val 12.0% | test 19.9%
Fold 3 leakage check: PASS
Fold 3 class_weight: {np.str_('bcc'): np.float64(1.165), np.str_('bkl'): np.float64(1.527), np.str_('df'): np.float64(17.796), np.str_('melanoma'): np.float64(0.892), np.str_('nevus'): np.float64(0.308), np.str_('vasc'): np.float64(15.282)}

===== Fold 3 setup verified. Training Custom CNN + EfficientNetB0. =====

FOLD 3: Custom CNN
Found 16230 validated image filenames belonging to 6 classes.
Found 2852 validated image filenames belonging to 6 classes.
Found 4754 validated image filenames belonging to 6 classes.
class_indices: {'bcc': 0, 'bkl': 1, 'df': 2, 'melanoma': 3, 'nevus': 4, 'vasc': 5}


I0000 00:00:1786892030.397901      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786892030.404181      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/60


E0000 00:00:1786892033.588640      24 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/dropout/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1786892035.237146      68 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1786892037.664429      66 service.cc:152] XLA service 0x7e6095374ae0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1786892037.664479      66 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1786892037.664486      66 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1786892037.940246      66 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


508/508 [==============================] - 682s 1s/step - loss: 1.7572 - accuracy: 0.2784 - val_loss: 1.5353 - val_accuracy: 0.4776
Epoch 2/60
508/508 [==============================] - 497s 978ms/step - loss: 1.6469 - accuracy: 0.3190 - val_loss: 1.3130 - val_accuracy: 0.5046
Epoch 3/60
508/508 [==============================] - 494s 973ms/step - loss: 1.5635 - accuracy: 0.3596 - val_loss: 2.6395 - val_accuracy: 0.1571
Epoch 4/60
508/508 [==============================] - 471s 928ms/step - loss: 1.5444 - accuracy: 0.3558 - val_loss: 1.3315 - val_accuracy: 0.4365
Epoch 5/60
508/508 [==============================] - 474s 932ms/step - loss: 1.5015 - accuracy: 0.3826 - val_loss: 1.3609 - val_accuracy: 0.4674
Epoch 6/60
508/508 [==============================] - 484s 952ms/step - loss: 1.5022 - accuracy: 0.3938 - val_loss: 1.6353 - val_accuracy: 0.3668
Epoch 7/60
508/508 [==============================] - 493s 971ms/step - loss: 1.4381 - accuracy: 0.4068 - val_loss: 1.5088 - val_accuracy:

E0000 00:00:1786896881.124020      24 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/efficientnetb0/block2b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


508/508 [==============================] - 418s 811ms/step - loss: 1.4036 - accuracy: 0.4551 - val_loss: 1.0759 - val_accuracy: 0.6168
Epoch 2/10
508/508 [==============================] - 415s 816ms/step - loss: 1.1444 - accuracy: 0.5330 - val_loss: 1.0416 - val_accuracy: 0.6220
Epoch 3/10
508/508 [==============================] - 398s 784ms/step - loss: 1.0186 - accuracy: 0.5633 - val_loss: 1.1376 - val_accuracy: 0.5617
Epoch 4/10
508/508 [==============================] - 391s 769ms/step - loss: 0.9755 - accuracy: 0.5764 - val_loss: 1.0484 - val_accuracy: 0.5827
Epoch 5/10
508/508 [==============================] - 375s 739ms/step - loss: 0.9271 - accuracy: 0.5816 - val_loss: 1.1378 - val_accuracy: 0.5589
Epoch 6/10
508/508 [==============================] - 380s 748ms/step - loss: 0.8897 - accuracy: 0.5929 - val_loss: 1.0361 - val_accuracy: 0.5887
Epoch 7/10
508/508 [==============================] - 371s 730ms/step - loss: 0.8358 - accuracy: 0.6079 - val_loss: 0.9483 - val_accura

E0000 00:00:1786900855.815664      24 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/efficientnetb0/block2b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


508/508 [==============================] - 523s 944ms/step - loss: 3.8827 - accuracy: 0.4022 - val_loss: 1.5508 - val_accuracy: 0.4975
Epoch 2/60
508/508 [==============================] - 477s 939ms/step - loss: 1.8998 - accuracy: 0.4419 - val_loss: 1.5065 - val_accuracy: 0.4996
Epoch 3/60
508/508 [==============================] - 479s 943ms/step - loss: 1.4106 - accuracy: 0.4635 - val_loss: 1.3785 - val_accuracy: 0.5266
Epoch 4/60
508/508 [==============================] - 466s 917ms/step - loss: 1.2471 - accuracy: 0.4951 - val_loss: 1.2506 - val_accuracy: 0.5708
Epoch 5/60
508/508 [==============================] - 466s 918ms/step - loss: 1.1013 - accuracy: 0.5205 - val_loss: 1.1783 - val_accuracy: 0.5852
Epoch 6/60
508/508 [==============================] - 464s 914ms/step - loss: 1.0252 - accuracy: 0.5500 - val_loss: 1.1078 - val_accuracy: 0.6045
Epoch 7/60
508/508 [==============================] - 475s 936ms/step - loss: 0.9445 - accuracy: 0.5677 - val_loss: 1.0730 - val_accura